# Zucchini Color Quality Scoring — Full Pipeline Overview

**Objective:** Automatically rank zucchini samples into 8 quality classes (Score 1 = poor → Score 8 = excellent) based on skin color, using a CNN with ordinal regression.

| Stage | Description |
|-------|------------|
| 1 | Raw image acquisition (16-bit TIFF) |
| 2 | Color correction via physical color checker |
| 3 | Circular cropping & inscribed square extraction |
| 4 | Statistical color filtering (LAB z-score) |
| 5 | White pixel inpainting (mean fill) |
| 6 | CIELAB conversion & feature extraction |
| 7 | Data augmentation & train/test split |
| 8 | CNN training (dual-head ordinal + regression) |
| 9 | Evaluation & confusion matrix |


## 1 · Imports & Setup

In [ ]:
import os, glob, random, warnings, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import tifffile as tiff
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from scipy.stats import spearmanr, kendalltau
from scipy.special import expit as sigmoid_np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, r2_score,
                             mean_squared_error, mean_absolute_error)

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
plt.style.use('seaborn-v0_8')

# Output directory
OUT_DIR = 'pipeline_overview_outputs'
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUT_DIR, 'model'), exist_ok=True)

def savefig(name, dpi=150):
    path = os.path.join(OUT_DIR, name)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'Saved: {path}')

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')


## 2 · Configuration

Key parameters used throughout the pipeline:


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
CC_ROOT       = r'F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform'
#CROPPED_ROOT  = r'F:\Aymen\server\cropped'
#FILTERED_ROOT = r'F:\Aymen\server\statistical_filtering'
OUT_DIR       = os.path.join(os.getcwd(), 'pipeline_overview_outputs')

# ── Model & training ──────────────────────────────────────────────────────────
IMG_SIZE     = (256, 256)
NUM_CLASSES  = 8
BATCH_SIZE   = 16
EPOCHS       = 400
TEST_SPLIT   = 0.20            # only used if you bypass CV
K_FOLDS      = 5               # stratified K-fold CV
PATIENCE_ES  = 30              # early stopping patience
PATIENCE_LR  = 8               # LR reduction patience
LR_INIT      = 5e-4
LR_MIN       = 1e-6

SCORE_FOLDERS = [f'Scorings={str(i).zfill(2)}' for i in range(1, NUM_CLASSES + 1)]
LABEL_MAP     = {name: idx for idx, name in enumerate(SCORE_FOLDERS)}
IDX_TO_SCORE  = np.array([1,2,3,4,5,6,7,8], dtype=np.float32)

# Create output sub-folders
for sub in ['', 'figures', 'figures/raw', 'figures/fill', 'model']:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

def savefig(name, subdir='figures', dpi=150):
    path = os.path.join(OUT_DIR, subdir, name)
    plt.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  saved: {path}')

#print(f'Cropped root  : {CROPPED_ROOT} | exists: {os.path.exists(CROPPED_ROOT)}')
#print(f'Filtered root : {FILTERED_ROOT} | exists: {os.path.exists(FILTERED_ROOT)}')
print(f'Output dir    : {OUT_DIR}')
print(f'K-fold        : {K_FOLDS} | ES patience: {PATIENCE_ES} | LR patience: {PATIENCE_LR}')


## 3 · Step 1 — Raw Image Acquisition & Color Correction

**Why color correction?** Raw images are captured with different lighting conditions and camera sensors. A physical **color checker** (X-Rite ColorChecker) is placed in each image to compute a color transformation matrix that normalizes device-specific biases.

**Process:**
1. Load raw 16-bit TIFF images
2. Detect color checker patches in the image
3. Compute correction matrix (reference vs detected patch colors)
4. Apply correction to the entire image
5. Extract the zucchini Region of Interest (ROI)

This step is handled by the `DD_CC_Task/extract_corrected_rois_raw.ipynb` notebook. The output is color-corrected images organized by score class (`Score=01` to `Score=08`).


## 4 · Step 2 — Rectangular Cropping

**Why?** The raw images contain background, equipment, and the color checker. We extract only the zucchini region using an inscribed square from a circular ROI.

**Method:**
1. Center = image center, radius = `min(H, W) / 2`
2. Inscribed square side = `2r / √2` → gives a 619×619 crop
3. Pixels outside the inscribed region → white (255, 255, 255)

This produces clean, centered zucchini images with consistent dimensions.


In [ ]:
pip install rawpy

In [ ]:
# Load and display cropped images (1 per score class)
import rawpy
import imageio.v2 as imageio

def load_image(path):
    ext = os.path.splitext(path)[1].lower()

    if ext == '.dng':
        with rawpy.imread(path) as raw:
            img = raw.postprocess()
        return img

    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)

    if img is None:
        img = tiff.imread(path)

        if img.ndim == 4:
            img = img[0]

        if img.ndim == 3 and img.shape[0] in (1, 3, 4):
            img = np.transpose(img, (1, 2, 0))

    if img.dtype == np.uint16:
        img = (img.astype(np.float32) / 65535.0 * 255.0).astype(np.uint8)

    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[-1] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    elif img.shape[-1] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    return img
def center_crop(img, crop_size=(619, 619)):
    h, w = img.shape[:2]
    ch, cw = crop_size

    # Ensure crop size does not exceed image dimensions
    ch = min(ch, h)
    cw = min(cw, w)

    start_y = (h - ch) // 2
    start_x = (w - cw) // 2

    return img[start_y:start_y + ch, start_x:start_x + cw]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Cropped Images — 1 Sample per Score Class',
             fontsize=14, fontweight='bold')

for score in range(1, NUM_CLASSES + 1):

    # Find folder like: 2026-02-26_Scorings_01
    matches = glob.glob(
        os.path.join(CC_ROOT, f'*_Scorings_{score:02d}')
    )

    if not matches:
        print(f'No folder found for score {score:02d}')
        continue

    folder = matches[0]

    files = sorted(glob.glob(os.path.join(folder, '*.dng')))
    if not files:
        print(f'No TIFF files in {folder}')
        continue

    img = load_image(files[4])
    img = center_crop(img, (619, 619))
    ax = axes[(score - 1) // 4, (score - 1) % 4]
    ax.imshow(img)
    ax.set_title(
        f'Score={score:02d} ',
        fontsize=10
    )
    ax.axis('off')

plt.tight_layout()
savefig('step2_cropped_samples.png')
plt.show()


## 5 · Step 3 — Statistical Color Filtering

**Why?** The cropped images still contain outlier pixels — netting artifacts, specular reflections, and edge noise that would confuse the CNN.

**Method:** For each image, we compute the mean (μ) and standard deviation (σ) of each CIELAB channel on foreground pixels only. Pixels outside the interval `[μ - 1.5σ, μ + 1.5σ]` are rejected and set to **white (255, 255, 255)**.

$$\text{keep} \iff \mu_X - 1.5\cdot\sigma_X \;\le\; X \;\le\; \mu_X + 1.5\cdot\sigma_X \quad \forall\; X \in \{L^*, a^*, b^*\}$$

This removes ~15-25% of foreground pixels that are color outliers.


In [ ]:
# Display filtered images vs cropped (before/after)
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(14, NUM_CLASSES * 3))
fig.suptitle('Statistical Filtering — Before vs After (K=1.5σ)', fontsize=14, fontweight='bold')

for score in range(1, NUM_CLASSES + 1):
    row = score - 1
    # Cropped (before filtering)
    cropped_files = sorted(glob.glob(os.path.join(CROPPED_ROOT, f'Score={score:02d}', '*.tif')))
    filtered_files = sorted(glob.glob(os.path.join(FILTERED_ROOT, f'Score={score:02d}', '*.tif')))

    if not cropped_files or not filtered_files: continue

    # Match by filename
    crop_name = os.path.basename(cropped_files[0])
    filt_match = [f for f in filtered_files if os.path.basename(f) == crop_name]
    if not filt_match:
        filt_match = [filtered_files[0]]
        crop_img = load_image(cropped_files[0])
    else:
        crop_img = load_image(cropped_files[0])

    filt_img = load_image(filt_match[0])

    # Compute white pixel mask (pixels that were filtered out)
    white_mask = np.all(filt_img == 255, axis=-1)

    axes[row, 0].imshow(crop_img)
    axes[row, 0].set_title(f'Score={score:02d} — Cropped', fontsize=9)
    axes[row, 0].axis('off')

    axes[row, 1].imshow(filt_img)
    n_white = white_mask.sum()
    n_total = filt_img.shape[0] * filt_img.shape[1]
    axes[row, 1].set_title(f'Filtered ({n_white/n_total*100:.1f}% white)', fontsize=9)
    axes[row, 1].axis('off')

    # Show the rejected pixels as red overlay on original
    overlay = crop_img.copy()
    overlay[white_mask] = [255, 80, 80]
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title('Rejected pixels (red)', fontsize=9)
    axes[row, 2].axis('off')

plt.tight_layout()
savefig('step3_statistical_filtering.png')
plt.show()


## 6 · Step 4 — White Pixel Inpainting (Mean Fill)

**Why?** The filtered images contain white pixels where outliers were removed. These white regions can bias the CNN — the network might learn to use white pixel density as a feature instead of actual color.

**Solution:** Replace all white pixels with the **mean color** of the non-white (valid) foreground pixels. This produces a uniform background that carries no discriminative signal, forcing the CNN to focus on the actual color distribution.


In [ ]:
def inpaint_white_with_mean(img_rgb):
    """Replace pure-white pixels (255,255,255) with mean colour of non-white pixels."""
    white_mask = np.all(img_rgb == 255, axis=-1)
    non_white  = img_rgb[~white_mask]
    if len(non_white) == 0:
        return img_rgb.copy()
    fill = non_white.mean(axis=0).astype(np.uint8)
    out  = img_rgb.copy()
    out[white_mask] = fill
    return out

def inpaint_white_with_median(img_rgb):
    """Replace pure-white pixels with the median colour of non-white pixels (robust to outliers)."""
    white_mask = np.all(img_rgb == 255, axis=-1)
    non_white  = img_rgb[~white_mask]
    if len(non_white) == 0:
        return img_rgb.copy()
    fill = np.median(non_white, axis=0).astype(np.uint8)
    out  = img_rgb.copy()
    out[white_mask] = fill
    return out


# ── Visualise: filtered → mean-inpainted → median-inpainted ──────────────────
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(16, NUM_CLASSES * 3))
fig.suptitle('White Pixel Inpainting — Mean vs Median Fill',
             fontsize=14, fontweight='bold')

for score in range(1, NUM_CLASSES + 1):
    row = score - 1
    filtered_files = sorted(glob.glob(os.path.join(FILTERED_ROOT, f'Score={score:02d}', '*.tif')))
    if not filtered_files: continue

    filt_img   = load_image(filtered_files[0])
    fill_mean  = inpaint_white_with_mean(filt_img)
    fill_med   = inpaint_white_with_median(filt_img)

    white_mask = np.all(filt_img == 255, axis=-1)
    non_white  = filt_img[~white_mask]
    mean_col   = non_white.mean(axis=0).astype(np.uint8)   if len(non_white) > 0 else np.array([128]*3)
    med_col    = np.median(non_white, axis=0).astype(np.uint8) if len(non_white) > 0 else np.array([128]*3)
    n_white_pct = 100.0 * white_mask.mean()

    axes[row, 0].imshow(filt_img);     axes[row, 0].set_title(f'S{score:02d} filtered ({n_white_pct:.1f}% white)', fontsize=9)
    axes[row, 1].imshow(fill_mean);    axes[row, 1].set_title(f'mean fill {tuple(mean_col)}', fontsize=9)
    axes[row, 2].imshow(fill_med);     axes[row, 2].set_title(f'median fill {tuple(med_col)}', fontsize=9)
    swatch_mean = np.full((100,100,3), mean_col, dtype=np.uint8)
    swatch_med  = np.full((100,100,3), med_col,  dtype=np.uint8)
    composite   = np.concatenate([swatch_mean, swatch_med], axis=1)
    axes[row, 3].imshow(composite); axes[row, 3].set_title('mean | median', fontsize=9)
    for ax in axes[row]: ax.axis('off')

plt.tight_layout()
savefig('step4_white_inpainting.png')
plt.show()


## 7 · Step 5 — CIELAB Conversion & Feature Extraction

**Why CIELAB instead of RGB?**
- The `a*` channel directly encodes the **green ↔ red** axis — the primary color quality signal for zucchini
- CIELAB is perceptually uniform: equal numerical differences correspond to equal perceived color differences (ΔE)
- RGB mixes color information across channels, making it harder for the CNN to isolate the quality signal

**Feature extraction (54 dimensions):**
For each LAB channel (L*, a*, b*): `mean (1) + std (1) + 16-bin histogram (16)` = 18 dims × 3 channels = **54 total**

These handcrafted features are injected into the CNN via late fusion to provide global color statistics that spatial pooling might dilute.


In [ ]:
def load_tif_rgb(path, size):
    img = tiff.imread(path)
    if img.ndim == 4: img = img[0]
    if img.ndim == 3 and img.shape[0] in (1, 3, 4):
        img = np.transpose(img, (1, 2, 0))
    if img.ndim == 2: img = np.stack([img]*3, axis=-1)
    elif img.shape[-1] == 1: img = np.concatenate([img]*3, axis=-1)
    elif img.shape[-1] == 4: img = img[..., :3]
    if img.dtype != np.uint8:
        img = img.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = (img * 255).astype(np.uint8)
    return cv2.resize(img, (size[1], size[0]), interpolation=cv2.INTER_AREA)

def rgb_to_lab(img_rgb):       return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
def lab_for_display(img_lab):  return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)

def extract_lab_features(img_lab_uint8):
    feats = []
    for c in range(3):
        ch = img_lab_uint8[:, :, c].astype(np.float32)
        feats.append(ch.mean() / 255.0)
        feats.append(ch.std()  / 255.0)
        hist, _ = np.histogram(ch, bins=16, range=(0, 256))
        hist = hist.astype(np.float32); hist /= (hist.sum() + 1e-8)
        feats.extend(hist.tolist())
    return np.array(feats, dtype=np.float32)


# ── Load raw filtered images ONCE, then derive the mean-filled variant ───────
def collect_raw(root, label_map, img_size):
    rgb_list, label_list, paths_list = [], [], []
    for score_name, label in label_map.items():
        paths = set()
        for ext in ['*.tif','*.TIF','*.tiff','*.TIFF']:
            paths.update(glob.glob(os.path.join(root, '**', score_name, ext), recursive=True))
        for p in sorted(paths):
            try:
                rgb_list.append(load_tif_rgb(p, img_size))
                label_list.append(label); paths_list.append(p)
            except Exception as e:
                print(f'  Warn {p}: {e}')
    print(f'Loaded {len(rgb_list)} raw filtered images')
    return (np.array(rgb_list, dtype=np.uint8),
            np.array(label_list, dtype=np.int32),
            paths_list)

X_rgb_raw, y_raw, paths = collect_raw(FILTERED_ROOT, LABEL_MAP, IMG_SIZE)
X_lab_raw  = np.stack([rgb_to_lab(r) for r in X_rgb_raw], axis=0)
feats_raw  = np.array([extract_lab_features(l) for l in X_lab_raw], dtype=np.float32)

# Mean-filled variant
X_rgb_fill = np.stack([inpaint_white_with_mean(r) for r in X_rgb_raw], axis=0)
X_lab_fill = np.stack([rgb_to_lab(r) for r in X_rgb_fill], axis=0)
feats_fill = np.array([extract_lab_features(l) for l in X_lab_fill], dtype=np.float32)

print(f'Raw     : RGB {X_rgb_raw.shape}  LAB {X_lab_raw.shape}  feats {feats_raw.shape}')
print(f'Filled  : RGB {X_rgb_fill.shape} LAB {X_lab_fill.shape} feats {feats_fill.shape}')
for i, s in enumerate(SCORE_FOLDERS):
    print(f'  {s}: {(y_raw == i).sum()} images')

# Aliases for back-compat with later cells that touch X_rgb / X_lab
X_rgb = X_rgb_raw
X_lab = X_lab_raw

DATASETS = {
    'raw' : dict(X_rgb=X_rgb_raw,  X_lab=X_lab_raw,  feats=feats_raw),
    'fill': dict(X_rgb=X_rgb_fill, X_lab=X_lab_fill, feats=feats_fill),
}


### LAB Channel Decomposition

Visualizing the three CIELAB channels shows that the `a*` channel provides a clear gradient from green (low scores) to red/yellow (high scores), confirming it is the primary discriminative feature.


In [ ]:
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(12, NUM_CLASSES * 2.2))
col_titles = ['RGB', 'L* (lightness)', 'a* (green↔red)', 'b* (blue↔yellow)']

for cls_idx in range(NUM_CLASSES):
    idxs = np.where(y_raw == cls_idx)[0]
    if len(idxs) == 0: continue
    rgb = X_rgb[idxs[0]]
    lab = X_lab[idxs[0]]

    axes[cls_idx, 0].imshow(rgb)
    axes[cls_idx, 0].set_ylabel(SCORE_FOLDERS[cls_idx], fontsize=9, fontweight='bold')
    for c, (cmap, title) in enumerate(zip(
            ['gray', 'RdYlGn_r', 'RdYlBu_r'], col_titles[1:])):
        axes[cls_idx, c+1].imshow(lab[:, :, c], cmap=cmap, vmin=0, vmax=255)

    for ax in axes[cls_idx]:
        ax.set_xticks([]); ax.set_yticks([])

for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=9, fontweight='bold')

plt.suptitle('CIELAB Channel Decomposition per Score Class', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
savefig('step5_lab_channels.png')
plt.show()


In [ ]:
# Feature distributions: key LAB stats per class
key_feat_idx  = [0, 18, 36, 1, 19, 37]
key_feat_name = ['L* mean', 'a* mean', 'b* mean', 'L* std', 'a* std', 'b* std']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, fi, name in zip(axes.ravel(), key_feat_idx, key_feat_name):
    per_class = [feats_raw[y_raw == c, fi] for c in range(NUM_CLASSES)]
    bp = ax.boxplot(per_class, patch_artist=True, notch=False)
    colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, NUM_CLASSES))
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    ax.set_xticklabels([f'S{i+1}' for i in range(NUM_CLASSES)], fontsize=8)
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('LAB Feature Distributions per Score Class', fontsize=14, fontweight='bold')
plt.tight_layout()
savefig('step5_feature_distributions.png')
plt.show()


## 8 · Step 6 — Data Augmentation

**Strategy:** Geometric transforms on RGB + photometric on LAB L-channel only.

| Type | Variants | Color-safe? |
|------|----------|------------|
| Flip | H / V / HV | Yes — geometry only |
| Rotation | ±15° / +30° | Yes — geometry only |
| Crop/Zoom | +10% / +20% / +30% | Yes — geometry only |
| Blur | σ=0.5 / 1.0 / 1.5 | Yes — applied to LAB |
| Noise | σ=3 / 6 / 9 | Yes — applied to LAB |

**Critical rule:** We never augment the `a*` or `b*` chrominance channels because they encode the color quality signal we're trying to measure. Augmenting them would destroy the ground truth.

**Dataset multiplier:** Each original image produces 15 augmented variants → ×16 total.


In [ ]:
# ── Augmentation functions ─────────────────────────────────────────────────────
def aug_flip(rgb, v):
    return cv2.flip(rgb, [1, 0, -1][v])

def aug_rotate(rgb, v):
    angles = [15, -15, 30]
    h, w = rgb.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angles[v], 1.0)
    return cv2.warpAffine(rgb, M, (w, h), borderMode=cv2.BORDER_REFLECT)

def aug_zoom(rgb, v):
    pcts = [0.10, 0.20, 0.30]
    f = pcts[v]
    h, w = rgb.shape[:2]
    dh, dw = int(h*f/2), int(w*f/2)
    return cv2.resize(rgb[dh:h-dh, dw:w-dw], (w, h), interpolation=cv2.INTER_LINEAR)

def aug_blur(lab, v):
    sigmas = [0.5, 1.0, 1.5]
    return cv2.GaussianBlur(lab, (3, 3), sigmas[v])

def aug_noise(lab, v):
    stds = [3, 6, 9]
    rng = np.random.default_rng(SEED + v)
    n = rng.normal(0, stds[v], lab.shape).astype(np.float32)
    return np.clip(lab.astype(np.float32) + n, 0, 255).astype(np.uint8)

AUG_CATALOGUE = [
    ('Flip',     aug_flip,   'rgb', 3),
    ('Rotation', aug_rotate, 'rgb', 3),
    ('Zoom',     aug_zoom,   'rgb', 3),
    ('Blur',     aug_blur,   'lab', 3),
    ('Noise',    aug_noise,  'lab', 3),
]

n_aug_variants = sum(n for *_, n in AUG_CATALOGUE)
print(f'Augmentation variants per image: {n_aug_variants}')
print(f'Dataset multiplier: x{1 + n_aug_variants}')


In [ ]:
# Augmentation grid: show all variants for one sample image
sample_idx = np.where(y_raw == 3)[0][0]  # pick Score=04 as example
orig_rgb = X_rgb[sample_idx]
orig_lab = X_lab[sample_idx]

aug_labels = ['Original',
              'Flip H', 'Flip V', 'Flip HV',
              'Rot +15', 'Rot -15', 'Rot +30',
              'Zoom 10%', 'Zoom 20%', 'Zoom 30%',
              'Blur 0.5', 'Blur 1.0', 'Blur 1.5',
              'Noise 3', 'Noise 6', 'Noise 9']

display_imgs = [orig_rgb]
for aug_name, aug_fn, domain, n_variants in AUG_CATALOGUE:
    for v in range(n_variants):
        if domain == 'rgb':
            display_imgs.append(aug_fn(orig_rgb, v))
        else:
            display_imgs.append(lab_for_display(aug_fn(orig_lab, v)))

fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for ax, img, label in zip(axes.ravel(), display_imgs, aug_labels):
    ax.imshow(img)
    ax.set_title(label, fontsize=8, fontweight='bold')
    ax.axis('off')

plt.suptitle('All Augmentation Variants (Score=04 sample)', fontsize=13, fontweight='bold')
plt.tight_layout()
savefig('step6_augmentation_grid.png')
plt.show()


## 9 · Step 7 — K-Fold Cross-Validation Setup

Instead of one 80/20 split, we use **stratified 5-fold CV**: every image is used
both for training (in 4 folds) and validation (in 1 fold). Augmentation is
applied **only to the training fold of each split** — the val fold stays clean.

Final metrics are computed on **out-of-fold (OOF) predictions** — one prediction
per image, produced by the model that did not see that image during training.


In [ ]:
from sklearn.model_selection import StratifiedKFold

def augment_indices(X_rgb_src, X_lab_src, feats_src, train_idx):
    """Build augmented arrays for the given training indices."""
    aug_lab, aug_feat, aug_y = [], [], []
    for i in train_idx:
        aug_lab.append(X_lab_src[i]); aug_feat.append(feats_src[i]); aug_y.append(y_raw[i])
    for aug_name, aug_fn, domain, n_variants in AUG_CATALOGUE:
        for v in range(n_variants):
            for i in train_idx:
                if domain == 'rgb':
                    aimg = rgb_to_lab(aug_fn(X_rgb_src[i], v))
                else:
                    aimg = aug_fn(X_lab_src[i], v)
                aug_lab.append(aimg)
                aug_feat.append(extract_lab_features(aimg))
                aug_y.append(y_raw[i])
    return (np.array(aug_lab,  dtype=np.uint8),
            np.array(aug_feat, dtype=np.float32),
            np.array(aug_y,    dtype=np.int32))


skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(np.arange(len(y_raw)), y_raw))
print(f'Stratified {K_FOLDS}-fold splits ready over {len(y_raw)} images')
for fi, (tr, va) in enumerate(fold_splits):
    print(f'  Fold {fi+1}: train={len(tr)}  val={len(va)}  '
          f'val class counts={np.bincount(y_raw[va], minlength=NUM_CLASSES).tolist()}')


## 10 · Step 8 — CNN Architecture (Dual-Head Ordinal + Regression)

The model uses a **dual-head** approach combining ordinal classification and regression:

**CNN branch (image):**
- 7 convolutional blocks (32 → 64 → 128 → 256 → 256 → 512 → 512)
- Each block: Conv2D + BatchNorm + ReLU + MaxPool + Dropout
- GlobalAveragePooling → 512-dim vector

**Feature branch (handcrafted):**
- 54-dim LAB stats → Dense(64) → Dense(32)

**Late fusion:** Concatenate CNN (512) + features (32) = 544-dim → Dense(128) → Dropout(0.35)

**Dual output heads:**
- **Ordinal head:** 7 sigmoid outputs for P(score > k), k=1..7
- **Regression head:** 1 linear output for continuous score

**Final prediction:** `0.7 × ordinal_score + 0.3 × regression_score`

**Composite loss:**

$$L_{total} = 1.0 \cdot L_{ordinal} + 0.5 \cdot L_{huber} + 0.5 \cdot L_{ranking}$$


In [ ]:
# ── Loss functions ─────────────────────────────────────────────────────────────
def within_acc(preds, targets, gap):
    return float(np.mean(np.abs(np.round(preds) - targets) <= gap))

@tf.function
def ordinal_bce(y_score, ord_logits):
    y_ord = tf.stack(
        [tf.cast(y_score > tf.cast(k+1, tf.float32), tf.float32) for k in range(7)],
        axis=1)
    return tf.reduce_mean(
        tf.nn.sigmoid_cross_entropy_with_logits(y_ord, ord_logits))

@tf.function
def huber_loss(y_score, reg_out, delta=1.0):
    pred = tf.squeeze(reg_out, axis=-1)
    return tf.keras.losses.Huber(delta=delta)(y_score, pred)

@tf.function
def ranking_loss(y_score, reg_out, margin=1.0):
    pred = tf.squeeze(reg_out, axis=-1)
    s_i = tf.expand_dims(y_score, 1)
    s_j = tf.expand_dims(y_score, 0)
    p_i = tf.expand_dims(pred, 1)
    p_j = tf.expand_dims(pred, 0)
    gap = s_i - s_j
    mask = tf.cast(tf.abs(gap) >= 1.0, tf.float32)
    loss = tf.maximum(0.0, margin - tf.sign(gap) * (p_i - p_j))
    return tf.reduce_sum(mask * loss) / (tf.reduce_sum(mask) + 1e-8)

@tf.function
def composite_loss(y_score, ord_logits, reg_out):
    l_ord  = ordinal_bce(y_score, ord_logits)
    l_hub  = huber_loss(y_score, reg_out)
    l_rank = ranking_loss(y_score, reg_out)
    return 1.0*l_ord + 0.5*l_hub + 0.5*l_rank, l_ord, l_hub, l_rank

def predict_score_np(ord_logits, reg_out):
    ord_score = 1.0 + sigmoid_np(ord_logits).sum(axis=1)
    reg_score = np.clip(reg_out.squeeze(-1), 1.0, 8.0)
    return 0.7 * ord_score + 0.3 * reg_score

print('Loss functions defined')

def composite_monitor(r2, rho, tau, acc1, val_loss, mae, rmse):
    """Weighted blend for early-stop / best-checkpoint selection (R² is primary)."""
    rho = 0.0 if np.isnan(rho) else rho
    tau = 0.0 if np.isnan(tau) else tau
    return (1.00 * r2
          + 0.50 * rho
          + 0.30 * tau
          + 0.30 * acc1
          - 0.10 * (val_loss / 10.0)
          - 0.20 * (mae / 4.0)
          - 0.20 * (rmse / 4.0))


In [ ]:
def build_model(img_shape=(256, 256, 3), feat_dim=54, num_ord=7):
    """ZucchiniCNN — 5 conv blocks + handcrafted feat fusion + dual head."""
    inp_img = layers.Input(shape=img_shape, name='lab_image')

    filters_list = [32, 64, 128, 256, 256]
    dropouts     = [0.20, 0.25, 0.25, 0.30, 0.30]

    x = inp_img
    for i, (f, d) in enumerate(zip(filters_list, dropouts), 1):
        x = layers.Conv2D(f, 3, padding='same', use_bias=False, name=f'B{i}_conv')(x)
        x = layers.BatchNormalization(name=f'B{i}_bn')(x)
        x = layers.Activation('relu', name=f'B{i}_relu')(x)
        x = layers.MaxPooling2D(2, name=f'B{i}_pool')(x)
        x = layers.Dropout(d, name=f'B{i}_drop')(x)

    cnn_out = layers.GlobalAveragePooling2D(name='GAP')(x)        # (B, 256)

    inp_feat = layers.Input(shape=(feat_dim,), name='handcrafted_feats')
    h = layers.Dense(64, activation='relu', name='feat_mlp1')(inp_feat)
    h = layers.Dense(32, activation='relu', name='feat_mlp2')(h)  # (B, 32)

    fused = layers.Concatenate(name='fusion')([cnn_out, h])       # (B, 288)
    fused = layers.Dense(128, use_bias=False, name='FC1')(fused)
    fused = layers.BatchNormalization(name='FC1_bn')(fused)
    fused = layers.Activation('relu', name='FC1_relu')(fused)
    fused = layers.Dropout(0.35, name='FC1_drop')(fused)

    ord_logits = layers.Dense(num_ord, name='ord_logits')(fused)
    reg_out    = layers.Dense(1,       name='reg_out')(fused)

    return models.Model(inputs=[inp_img, inp_feat],
                        outputs=[ord_logits, reg_out],
                        name='ZucchiniCNN_5blk')

_probe = build_model()
_probe.summary(print_fn=lambda x: print(x))
print(f'\nTotal params: {_probe.count_params():,}')
del _probe


## 11 · K-Fold Training Function

We wrap the whole training loop in `run_kfold(name, X_rgb_src, X_lab_src, feats_src)`
so we can call it twice — once for the raw dataset and once for the mean-filled
dataset — and directly compare results.


In [ ]:
def make_dataset(X_img, X_feat, y_score, batch_size, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        ((X_img.astype(np.float32), X_feat.astype(np.float32)),
          y_score.astype(np.float32)))
    if shuffle:
        ds = ds.shuffle(len(X_img), seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)


def run_kfold(name, X_rgb_src, X_lab_src, feats_src):
    """Run stratified K-fold CV; returns dict with OOF preds + per-fold metrics + hist list."""
    print('\n' + '='*72)
    print(f'  RUN_KFOLD: dataset = "{name}"'.center(72))
    print('='*72)

    oof_pred = np.full(len(y_raw), np.nan, dtype=np.float32)
    oof_ord  = np.full((len(y_raw), 7), np.nan, dtype=np.float32)
    fold_metrics, fold_hist, fold_best_paths = [], [], []
    t_all = time.time()

    for fi, (tr_idx, va_idx) in enumerate(fold_splits):
        print(f'\n--- {name}  Fold {fi+1}/{K_FOLDS} ---')

        X_lab_tr_a, X_feat_tr_a, y_lbl_tr_a = augment_indices(
            X_rgb_src, X_lab_src, feats_src, tr_idx)
        X_img_tr = X_lab_tr_a.astype(np.float32) / 255.0
        y_tr_s   = IDX_TO_SCORE[y_lbl_tr_a]

        X_img_va  = X_lab_src[va_idx].astype(np.float32) / 255.0
        X_feat_va = feats_src[va_idx].astype(np.float32)
        y_va_s    = IDX_TO_SCORE[y_raw[va_idx]]

        train_ds = make_dataset(X_img_tr, X_feat_tr_a, y_tr_s, BATCH_SIZE, shuffle=True)
        val_ds   = make_dataset(X_img_va, X_feat_va, y_va_s, BATCH_SIZE, shuffle=False)

        tf.keras.backend.clear_session()
        random.seed(SEED + fi); np.random.seed(SEED + fi); tf.random.set_seed(SEED + fi)
        mdl = build_model(img_shape=(*IMG_SIZE, 3))
        opt = optimizers.Adam(learning_rate=LR_INIT)

        @tf.function
        def _tstep(x_img, x_feat, y_s):
            with tf.GradientTape() as tape:
                ol, ro = mdl([x_img, x_feat], training=True)
                tot, *_ = composite_loss(y_s, ol, ro)
            grads = tape.gradient(tot, mdl.trainable_variables)
            opt.apply_gradients(zip(grads, mdl.trainable_variables))
            return tot

        @tf.function
        def _vstep(x_img, x_feat):
            return mdl([x_img, x_feat], training=False)

        hist = {k:[] for k in ['train_loss','val_loss','val_spearman','val_kendall',
                                'val_mae','val_mse','val_rmse','val_r2',
                                'val_acc0','val_acc1','val_acc2','lr']}
        best_score, wait_es, wait_lr, best_ep = -np.inf, 0, 0, 0
        best_path = os.path.join(OUT_DIR, 'model', f'best_{name}_fold{fi+1}.keras')

        t0 = time.time()
        for ep in range(EPOCHS):
            losses = []
            for (xi, xf), ys in train_ds:
                losses.append(float(_tstep(xi, xf, ys)))
            tr_loss = float(np.mean(losses))

            ords, regs, ys_v = [], [], []
            for (xi, xf), ys in val_ds:
                o, r = _vstep(xi, xf); ords.append(o.numpy()); regs.append(r.numpy()); ys_v.append(ys.numpy())
            ord_v = np.concatenate(ords); reg_v = np.concatenate(regs); y_v = np.concatenate(ys_v)
            val_loss = float(composite_loss(
                tf.constant(y_v[:BATCH_SIZE]), tf.constant(ord_v[:BATCH_SIZE]),
                tf.constant(reg_v[:BATCH_SIZE]))[0])

            pred  = predict_score_np(ord_v, reg_v)
            rho,_ = spearmanr(y_v, pred); tau,_ = kendalltau(y_v, pred)
            mae   = float(mean_absolute_error(y_v, pred))
            mse   = float(mean_squared_error(y_v, pred));  rmse = float(np.sqrt(mse))
            r2    = float(r2_score(y_v, pred))
            acc0  = within_acc(pred, y_v, gap=0)
            acc1  = within_acc(pred, y_v, gap=1)
            acc2  = within_acc(pred, y_v, gap=2)
            cur_lr = float(opt.learning_rate)

            hist['train_loss'].append(tr_loss); hist['val_loss'].append(val_loss)
            hist['val_spearman'].append(rho);   hist['val_kendall'].append(tau)
            hist['val_mae'].append(mae);        hist['val_mse'].append(mse)
            hist['val_rmse'].append(rmse);      hist['val_r2'].append(r2)
            hist['val_acc0'].append(acc0);      hist['val_acc1'].append(acc1); hist['val_acc2'].append(acc2)
            hist['lr'].append(cur_lr)

            score = composite_monitor(r2, rho, tau, acc1, val_loss, mae, rmse)
            mark = ''
            if score > best_score:
                best_score = score; best_ep = ep; wait_es = wait_lr = 0
                mdl.save(best_path); mark = f'  BEST(score={score:.3f})'
            else:
                wait_es += 1; wait_lr += 1
            if wait_lr >= PATIENCE_LR:
                new_lr = max(cur_lr * 0.5, LR_MIN); opt.learning_rate.assign(new_lr); wait_lr = 0
                mark += f'  LR->{new_lr:.1e}'
            if (ep+1) % 5 == 0 or mark:
                print(f'  ep{ep+1:3d} loss={tr_loss:.3f} vl={val_loss:.3f} rho={rho:.3f} '
                      f'R2={r2:.3f} MAE={mae:.3f} Acc1={acc1*100:.0f}%{mark}')
            if wait_es >= PATIENCE_ES:
                print(f'  Early stop @ ep {ep+1}. Best ep {best_ep+1}'); break

        print(f'  Fold {fi+1} done in {(time.time()-t0)/60:.1f} min')

        # OOF
        mdl.load_weights(best_path)
        ords, regs = [], []
        for (xi, xf), _ in val_ds:
            o, r = _vstep(xi, xf); ords.append(o.numpy()); regs.append(r.numpy())
        ord_va = np.concatenate(ords); reg_va = np.concatenate(regs)
        pv = predict_score_np(ord_va, reg_va)
        oof_pred[va_idx] = pv; oof_ord[va_idx] = ord_va

        fold_hist.append(hist); fold_best_paths.append(best_path)
        fold_metrics.append({
            'fold': fi+1, 'best_epoch': best_ep+1, 'best_score': best_score,
            'rho': float(spearmanr(y_va_s, pv)[0]),
            'tau': float(kendalltau(y_va_s, pv)[0]),
            'r2':  float(r2_score(y_va_s, pv)),
            'mae': float(mean_absolute_error(y_va_s, pv)),
            'rmse': float(np.sqrt(mean_squared_error(y_va_s, pv))),
            'acc0': within_acc(pv, y_va_s, gap=0),
            'acc1': within_acc(pv, y_va_s, gap=1),
            'acc2': within_acc(pv, y_va_s, gap=2),
        })

    print(f'\n[{name}] ALL FOLDS DONE in {(time.time()-t_all)/60:.1f} min')
    return dict(name=name, oof_pred=oof_pred, oof_ord=oof_ord,
                fold_metrics=pd.DataFrame(fold_metrics),
                fold_hist=fold_hist, fold_best_paths=fold_best_paths)


## 12 · Training Curves — Per-Fold Overlay (Both Methods)

In [ ]:
def plot_fold_curves(results, ax_row, color_map, title_prefix=''):
    for fi, h in enumerate(results['fold_hist']):
        ep = np.arange(1, len(h['val_r2'])+1)
        ax_row[0].plot(ep, h['val_r2'],       color=color_map(fi), lw=1.5, alpha=0.85, label=f'F{fi+1}')
        ax_row[1].plot(ep, h['val_spearman'], color=color_map(fi), lw=1.5, alpha=0.85)
        ax_row[2].plot(ep, h['val_mae'],      color=color_map(fi), lw=1.5, alpha=0.85)
        ax_row[3].plot(ep, [v*100 for v in h['val_acc1']], color=color_map(fi), lw=1.5, alpha=0.85)
    for ax, ttl in zip(ax_row,
        [f'{title_prefix}R²', f'{title_prefix}Spearman ρ',
         f'{title_prefix}MAE', f'{title_prefix}Within-1 Acc (%)']):
        ax.set_title(ttl, fontweight='bold'); ax.set_xlabel('Epoch'); ax.grid(alpha=0.3)
    ax_row[0].set_ylim(-0.5, 1.05); ax_row[1].set_ylim(-0.1, 1.05); ax_row[3].set_ylim(0, 105)
    ax_row[0].legend(fontsize=7, ncol=2)

fig, axes = plt.subplots(2, 4, figsize=(22, 10))
plot_fold_curves(results_raw,  axes[0], plt.cm.Blues,   title_prefix='RAW · ')
plot_fold_curves(results_fill, axes[1], plt.cm.Oranges, title_prefix='FILL · ')

plt.suptitle(f'K-Fold Training Curves — Raw vs Mean-Fill',
             fontsize=15, fontweight='bold')
plt.tight_layout()
savefig('step8_training_curves_compare.png')
plt.show()


## 13 · OOF Evaluation — Both Methods

In [ ]:
def oof_metrics(res):
    y_true = IDX_TO_SCORE[y_raw]; pred = res['oof_pred']
    m = {
        'name'  : res['name'],
        'rho'   : float(spearmanr(y_true, pred)[0]),
        'tau'   : float(kendalltau(y_true, pred)[0]),
        'r2'    : float(r2_score(y_true, pred)),
        'mae'   : float(mean_absolute_error(y_true, pred)),
        'mse'   : float(mean_squared_error(y_true, pred)),
        'rmse'  : float(np.sqrt(mean_squared_error(y_true, pred))),
        'acc0'  : within_acc(pred, y_true, gap=0),
        'acc1'  : within_acc(pred, y_true, gap=1),
        'acc2'  : within_acc(pred, y_true, gap=2),
    }
    return m

oof_raw  = oof_metrics(results_raw)
oof_fill = oof_metrics(results_fill)
oof_df   = pd.DataFrame([oof_raw, oof_fill]).set_index('name')
print('\n' + '='*72)
print('  OUT-OF-FOLD EVALUATION — RAW vs MEAN-FILL'.center(72))
print('='*72)
print(oof_df.round(4).to_string())
print('='*72)

# Decide winner per metric (higher better for rho/tau/r2/acc*, lower for mae/mse/rmse)
higher_better = ['rho','tau','r2','acc0','acc1','acc2']
lower_better  = ['mae','mse','rmse']
print('\nWinner per metric:')
for m in higher_better:
    w = 'fill' if oof_df.loc['fill', m] > oof_df.loc['raw', m] else 'raw'
    delta = oof_df.loc['fill', m] - oof_df.loc['raw', m]
    print(f'  {m:>6}: {w}  (Δ = {delta:+.4f})')
for m in lower_better:
    w = 'fill' if oof_df.loc['fill', m] < oof_df.loc['raw', m] else 'raw'
    delta = oof_df.loc['fill', m] - oof_df.loc['raw', m]
    print(f'  {m:>6}: {w}  (Δ = {delta:+.4f})')

oof_df.to_csv(os.path.join(OUT_DIR, 'oof_metrics_compare.csv'))
print(f'\nSaved → {os.path.join(OUT_DIR, "oof_metrics_compare.csv")}')


In [ ]:
y_true_all = IDX_TO_SCORE[y_raw]
label_all  = y_raw

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
for row, (key, res) in enumerate(results_all.items()):
    pred = res['oof_pred']
    rho_v = oof_df.loc[key,'rho']; mae_v = oof_df.loc[key,'mae']

    # scatter
    ax = axes[row,0]
    jit = np.random.normal(0, 0.07, size=y_true_all.shape)
    sc = ax.scatter(y_true_all + jit, pred, c=np.abs(pred-y_true_all),
                    cmap='RdYlGn_r', vmin=0, vmax=2, alpha=0.6, s=22, edgecolors='none')
    ax.plot([1,8],[1,8],'k--',lw=1.5,label='perfect')
    ax.fill_between([1,8],[0,7],[2,9], alpha=0.12, color='green', label='±1')
    plt.colorbar(sc, ax=ax, label='|err|')
    ax.set_xlabel('True'); ax.set_ylabel('Pred')
    ax.set_title(f'[{key}] Pred vs True (ρ={rho_v:.3f}, MAE={mae_v:.3f})', fontweight='bold')
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

    # error hist
    ax = axes[row,1]
    err = pred - y_true_all
    ax.hist(err, bins=25, color='steelblue' if key=='raw' else 'darkorange',
            edgecolor='white', alpha=0.85)
    ax.axvline(0, color='red', lw=1.5)
    ax.set_title(f'[{key}] Error  μ={err.mean():.3f}  σ={err.std():.3f}', fontweight='bold')
    ax.set_xlabel('pred − true'); ax.grid(alpha=0.3)

    # per-class boxplot
    ax = axes[row,2]
    per_cls = [pred[label_all == c] for c in range(NUM_CLASSES)]
    bp = ax.boxplot(per_cls, patch_artist=True)
    cols = plt.cm.RdYlGn(np.linspace(0.1, 0.9, NUM_CLASSES))
    for patch, color in zip(bp['boxes'], cols): patch.set_facecolor(color)
    ax.plot(range(1,9), range(1,9), 'k--', lw=1.5, label='true')
    ax.set_xticklabels([f'S{i+1}' for i in range(NUM_CLASSES)], fontsize=8)
    ax.set_title(f'[{key}] Per-Class Predictions', fontweight='bold')
    ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Prediction Analysis — Raw (top) vs Mean-Fill (bottom)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
savefig('step9_prediction_analysis_compare.png')
plt.show()


## 14 · Per-Class Stats + Confusion Matrix (Both Methods)

In [ ]:
def per_class_stats(res):
    pred = res['oof_pred']
    rows = []
    print(f'\n--- Per-class stats [{res["name"]}] ---')
    for c in range(NUM_CLASSES):
        mask = label_all == c
        n = int(mask.sum())
        p = pred[mask]; t = y_true_all[mask]
        mp = float(p.mean()); sp = float(p.std()); mae_c = float(np.mean(np.abs(p - t)))
        rows.append((c+1, n, mp, sp, mae_c))
        print(f'  Class {c+1}: count={n:4d}  mean_pred={mp:.3f} ± {sp:.3f}  MAE={mae_c:.3f}')
    df = pd.DataFrame(rows, columns=['class','count','mean_pred','std_pred','mae'])
    df.to_csv(os.path.join(OUT_DIR, f'per_class_stats_{res["name"]}.csv'), index=False)
    return df

stats_raw  = per_class_stats(results_raw)
stats_fill = per_class_stats(results_fill)

# Confusion matrices side-by-side
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for row, (key, res) in enumerate(results_all.items()):
    pred_round = np.clip(np.round(res['oof_pred']).astype(int), 1, 8)
    true_int   = y_true_all.astype(int)
    cm = confusion_matrix(true_int, pred_round, labels=list(range(1, 9)))
    cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues' if key=='raw' else 'Oranges',
                xticklabels=[f'S{i}' for i in range(1,9)],
                yticklabels=[f'S{i}' for i in range(1,9)],
                ax=axes[row,0], cbar_kws={'label':'count'})
    axes[row,0].set_title(f'[{key}] Confusion Matrix — Counts', fontweight='bold')
    axes[row,0].set_xlabel('Pred'); axes[row,0].set_ylabel('True')

    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens', vmin=0, vmax=1,
                xticklabels=[f'S{i}' for i in range(1,9)],
                yticklabels=[f'S{i}' for i in range(1,9)],
                ax=axes[row,1], cbar_kws={'label':'recall'})
    axes[row,1].set_title(f'[{key}] Confusion Matrix — Row-Normalised', fontweight='bold')
    axes[row,1].set_xlabel('Pred'); axes[row,1].set_ylabel('True')

plt.suptitle('OOF Confusion Matrices — Raw (top) vs Mean-Fill (bottom)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
savefig('step9_confusion_matrix_compare.png')
plt.show()


In [ ]:
def prediction_frieze(res, X_lab_src, title):
    pred = res['oof_pred']
    order = np.argsort(y_true_all)
    sub   = order[::3]
    cls_seq = y_true_all[sub].astype(int)
    bounds  = [0] + [i for i in range(1, len(sub)) if cls_seq[i] != cls_seq[i-1]] + [len(sub)]
    n_panels = len(sub) + 9
    fig, axes = plt.subplots(1, n_panels, figsize=((n_panels) * 1.6, 6))

    pi = 0
    for g in range(len(bounds)-1):
        gs, ge = bounds[g], bounds[g+1]; cls = cls_seq[gs]
        if pi < n_panels:
            ax = axes[pi]; ax.axis('off')
            ax.text(0.5, 0.5, f'S{cls+1}', ha='center', va='center',
                    fontsize=22, fontweight='bold',
                    color=plt.cm.RdYlGn(cls / (NUM_CLASSES-1)))
            pi += 1
        for i in range(gs, ge):
            if pi >= n_panels: break
            idx = sub[i]; ax = axes[pi]
            ax.imshow(lab_for_display(X_lab_src[idx]))
            ts = int(y_true_all[idx]); ps = pred[idx]; e = abs(ps - ts)
            col = 'limegreen' if e <= 0.5 else ('orange' if e <= 1.0 else 'red')
            ax.set_title(f'T:{ts} P:{ps:.1f}', fontsize=8, color=col, fontweight='bold')
            ax.set_xticks([]); ax.set_yticks([])
            for sp in ax.spines.values(): sp.set_edgecolor(col); sp.set_linewidth(1.5)
            pi += 1
    while pi < n_panels: axes[pi].axis('off'); pi += 1

    plt.suptitle(title, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    savefig(f'step9_frieze_{res["name"]}.png')
    plt.show()


prediction_frieze(results_raw,  X_lab_raw,
                  'OOF Prediction Frieze — RAW (no white-fill)  '
                  '🟢 err≤0.5  🟠 err≤1  🔴 err>1')
prediction_frieze(results_fill, X_lab_fill,
                  'OOF Prediction Frieze — MEAN-FILL  '
                  '🟢 err≤0.5  🟠 err≤1  🔴 err>1')


## 15 · Method Comparison Summary — Raw vs Mean-Fill

We trained two CNN ensembles with identical architecture and training schedule:

- **raw**:  zucchini images on white background, white pixels left untouched
- **fill**: white pixels replaced with mean of non-white pixels (per image)

Both were trained with stratified 5-fold CV (early-stop patience 30, LR-reduce
patience 8). The table below reports out-of-fold metrics on the full dataset.


In [ ]:
print('=' * 72)
print('  METHOD COMPARISON — OOF METRICS'.center(72))
print('=' * 72)
print(oof_df.round(4).to_string())
print('=' * 72)

# Highlight headline metrics
for m in ['r2', 'rho', 'mae', 'acc1']:
    raw_v  = oof_df.loc['raw',  m]
    fill_v = oof_df.loc['fill', m]
    better = 'fill' if (fill_v > raw_v if m in ('r2','rho','acc1') else fill_v < raw_v) else 'raw'
    print(f'  {m:>5}:  raw={raw_v:.4f}  |  fill={fill_v:.4f}  →  {better.upper()} wins')

print('=' * 72)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
high = ['r2','rho','tau','acc0','acc1','acc2']
low  = ['mae','rmse']
x = np.arange(len(high))
axes[0].bar(x - 0.2, oof_df.loc['raw',  high].values, 0.4, label='raw',  color='steelblue')
axes[0].bar(x + 0.2, oof_df.loc['fill', high].values, 0.4, label='fill', color='darkorange')
axes[0].set_xticks(x); axes[0].set_xticklabels(high); axes[0].set_ylim(0, 1.05)
axes[0].set_title('Higher is better', fontweight='bold'); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
for i, m in enumerate(high):
    axes[0].text(i-0.2, oof_df.loc['raw',  m]+0.01, f'{oof_df.loc["raw",  m]:.3f}', ha='center', fontsize=8)
    axes[0].text(i+0.2, oof_df.loc['fill', m]+0.01, f'{oof_df.loc["fill", m]:.3f}', ha='center', fontsize=8)

x = np.arange(len(low))
axes[1].bar(x - 0.2, oof_df.loc['raw',  low].values, 0.4, label='raw',  color='steelblue')
axes[1].bar(x + 0.2, oof_df.loc['fill', low].values, 0.4, label='fill', color='darkorange')
axes[1].set_xticks(x); axes[1].set_xticklabels(low)
axes[1].set_title('Lower is better', fontweight='bold'); axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)
for i, m in enumerate(low):
    axes[1].text(i-0.2, oof_df.loc['raw',  m]+0.005, f'{oof_df.loc["raw",  m]:.3f}', ha='center', fontsize=8)
    axes[1].text(i+0.2, oof_df.loc['fill', m]+0.005, f'{oof_df.loc["fill", m]:.3f}', ha='center', fontsize=8)

plt.suptitle('Side-by-Side Comparison — Raw vs Mean-Fill (OOF)', fontsize=14, fontweight='bold')
plt.tight_layout()
savefig('step10_comparison_bars.png')
plt.show()
